In [ ]:
# --- DIMER Kaggle executor preamble (NOT part of the tutorial) ---
# Fetches the COMMITTED notebook bytes of zoedepth-metric-depth-pipeline@2dc73837422b0751fea79414f3de71f61ac800ec and runs them in a fresh IPython kernel (nbclient).
import hashlib, json, os, platform, shutil, subprocess, sys, time, urllib.request

REPO, SHA, NB_REL, EXPECTED_BLOB = "zoedepth-metric-depth-pipeline", "2dc73837422b0751fea79414f3de71f61ac800ec", "tutorials/zoedepth_metric_depth_colab.ipynb", "ee943c717635036ae64974e38ae15c71aecd8fb0"
SCRATCH, WORK = "/tmp/dimer-exec", "/content"
shutil.rmtree(SCRATCH, ignore_errors=True); os.makedirs(SCRATCH)
shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)

url = f"https://raw.githubusercontent.com/kurtvalcorza/{REPO}/{SHA}/{NB_REL}"
raw = urllib.request.urlopen(url, timeout=60).read()
blob = hashlib.sha1(b"blob %d\0" % len(raw) + raw).hexdigest()
assert blob == EXPECTED_BLOB, f"fetched notebook blob {blob} != committed {EXPECTED_BLOB}"

NB_PATH = os.path.join(SCRATCH, "tutorial.ipynb")
open(NB_PATH, "wb").write(raw)
open(os.path.join(SCRATCH, "runner.py"), "w", encoding="utf-8", newline="\n").write('import json, os, sys, time, traceback\nfrom pathlib import Path\nimport nbformat\nfrom nbclient import NotebookClient\n\nsrc, dst, rep, timeout = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]), int(sys.argv[4])\nRESTART_MARK = "Restart the runtime, then rerun from the top"\nr = {"ok": False, "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "passes": []}\nt0 = time.time()\n\nfor attempt in (1, 2):\n    # Each pass is a brand-new IPython kernel: exactly what the notebook\'s own\n    # "Restart the runtime, then rerun from the top" instruction asks a human to do.\n    nb = nbformat.read(src, as_version=4)\n    c = NotebookClient(nb, timeout=timeout, kernel_name="python3", allow_errors=False, resources={"metadata": {"path": os.getcwd()}})\n    p = {"attempt": attempt, "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n    t1 = time.time()\n    try:\n        c.execute()\n        r["ok"] = True\n        p["ok"] = True\n    except BaseException as e:\n        p["ok"] = False\n        p["error"] = f"{type(e).__name__}: {str(e)[-3000:]}"\n        r["error"] = p["error"]\n        r["traceback"] = traceback.format_exc()[-4000:]\n    p["wall_s"] = round(time.time() - t1, 1)\n    r["passes"].append(p)\n    nbformat.write(nb, dst if (p["ok"] or attempt == 2) else dst.with_name("executed-pass1.ipynb"))\n    if p["ok"] or RESTART_MARK not in p.get("error", ""):\n        break\n    r["restarted_after_install_cell"] = True\n\nr["wall_s"] = round(time.time() - t0, 1)\nr["finished"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())\nrep.write_text(json.dumps(r, indent=1), encoding="utf-8")\nprint("RUNNER", "OK" if r["ok"] else "FAILED", r["wall_s"], "s", "passes", len(r["passes"]))\nsys.exit(0 if r["ok"] else 1)\n')
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbclient==0.10.2", "nbformat"], check=True, capture_output=True)

def _v(d):
    import importlib.metadata as md
    try: return md.version(d)
    except md.PackageNotFoundError: return None

IDENTITY = {
    "repository": REPO, "candidate_sha": SHA, "notebook": NB_REL, "committed_blob": EXPECTED_BLOB, "fetched_blob_verified": True,
    "kaggle_docker_image": os.environ.get("KAGGLE_DOCKER_IMAGE"), "kaggle_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
    "os": platform.platform(), "python": platform.python_version(), "cpu_count": os.cpu_count(),
    "image_versions_before_run": {d: _v(d) for d in ("torch", "numpy", "transformers", "tokenizers", "safetensors", "huggingface-hub", "pillow", "nbclient", "ipykernel")},
    "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}
try:
    IDENTITY["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or None
except FileNotFoundError:
    IDENTITY["nvidia_smi"] = None

hub = os.path.expanduser("~/.cache/huggingface/hub")
IDENTITY["hf_cache_clean_at_start"] = (not os.path.exists(hub)) or not os.listdir(hub)
CHILD_ENV = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
CHILD_ENV.pop("DIMER_NOTEBOOK_CI_PREINSTALLED", None)
print(json.dumps(IDENTITY, indent=1))


In [ ]:
# --- DIMER Kaggle executor: run the notebook in a fresh IPython kernel (NOT part of the tutorial) ---
EXECUTED, RUNNER_REPORT = os.path.join(SCRATCH, "executed.ipynb"), os.path.join(SCRATCH, "runner-report.json")
proc = subprocess.run([sys.executable, os.path.join(SCRATCH, "runner.py"), NB_PATH, EXECUTED, RUNNER_REPORT, "3600"], env=CHILD_ENV, cwd=WORK)
print("[executor] runner exit", proc.returncode)


In [ ]:
# --- DIMER Kaggle executor evidence cell (NOT part of the tutorial) ---
EV = "/kaggle/working/evidence"
shutil.rmtree(EV, ignore_errors=True)
os.makedirs(os.path.join(EV, "outputs"))

report = json.load(open(RUNNER_REPORT, encoding="utf-8")) if os.path.exists(RUNNER_REPORT) else {"ok": False, "error": "runner produced no report"}
_p1 = os.path.join(SCRATCH, "executed-pass1.ipynb")
executed = json.load(open(EXECUTED if os.path.exists(EXECUTED) else _p1, encoding="utf-8")) if (os.path.exists(EXECUTED) or os.path.exists(_p1)) else None
if executed and os.path.exists(EXECUTED): shutil.copy(EXECUTED, os.path.join(EV, "executed.ipynb"))
if os.path.exists(_p1): shutil.copy(_p1, os.path.join(EV, "executed-pass1.ipynb"))

cells = []
if executed:
    for i, c in enumerate(executed["cells"]):
        if c.get("cell_type") != "code": continue
        outs = c.get("outputs", [])
        text = "".join("".join(o.get("text", "")) if o.get("output_type") == "stream" else "".join(o.get("data", {}).get("text/plain", "")) if o.get("output_type") in ("execute_result", "display_data") else "" for o in outs)
        err = next((o for o in outs if o.get("output_type") == "error"), None)
        ex = c.get("metadata", {}).get("execution", {})
        cells.append({
            "cell": i, "status": "error" if err else ("ok" if c.get("execution_count") is not None else "not-run"),
            "started": ex.get("iopub.execute_input"), "ended": ex.get("shell.execute_reply"), "stdout_tail": text[-1500:],
            "error": (err["ename"] + ": " + err["evalue"][:1500]) if err else None
        })

preserved = {}
outdir = os.path.join(WORK, "outputs")
if os.path.isdir(outdir):
    for root, _d, names in os.walk(outdir):
        for n in sorted(names):
            p = os.path.join(root, n); rel = os.path.relpath(p, outdir); size = os.path.getsize(p)
            preserved[rel] = {"bytes": size, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest()}
            if size <= 50_000_000:
                dst = os.path.join(EV, "outputs", rel); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(p, dst)

weights = []
wdir = os.path.join(WORK, "weights")
if os.path.isdir(wdir):
    for root, _d, names in os.walk(wdir):
        for n in sorted(names):
            p = os.path.join(root, n); weights.append({"path": os.path.relpath(p, wdir), "bytes": os.path.getsize(p)})

try:
    import torch
    tv = {"torch": torch.__version__, "cuda": torch.version.cuda, "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
except Exception as e:
    tv = repr(e)

summary = {
    "identity": IDENTITY, "runner": report, "cells": cells, "preserved_outputs": preserved,
    "staged_weights": weights, "executor_torch_after_run": tv, "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}
json.dump(summary, open(os.path.join(EV, "run_summary.json"), "w", encoding="utf-8"), indent=1, ensure_ascii=False)
print(json.dumps({k: v for k, v in summary.items() if k not in ("cells",)}, indent=1, ensure_ascii=False))
for c in cells:
    print(f"cell {c['cell']}: {c['status']} {c['started']} -> {c['ended']}" + (f"  ERROR {c['error']}" if c["error"] else ""))

shutil.rmtree(SCRATCH, ignore_errors=True); shutil.rmtree(WORK, ignore_errors=True)
assert report.get("ok"), "notebook execution FAILED; see cells above"
